# Variant Pathogenicity DL — Analysis

Post-run exploration of the artifacts produced by `main.py`:

* out-of-fold (OOF) predictions with uncalibrated / temperature / isotonic / PLLR scores,
* pooled discrimination & calibration metrics per score variant,
* reliability diagrams,
* held-out ClinVar VUS risk predictions.

Run the pipeline first, e.g. `python main.py --gene TP53`, then execute this notebook.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, str(Path.cwd()))
from src.calibration import full_report, plot_reliability_diagrams

sns.set_theme(style="whitegrid")

GENE = "TP53"                     # <-- change to the gene you trained
PROCESSED = Path("data") / "processed"

oof = pd.read_csv(PROCESSED / f"{GENE}_oof_predictions.csv")
vus = pd.read_csv(PROCESSED / f"{GENE}_vus_predictions.csv")
fold_metrics = pd.read_csv(PROCESSED / f"{GENE}_fold_metrics.csv")
print(oof.shape, vus.shape)
oof.head()

## Pooled discrimination & calibration comparison

In [ ]:
score_cols = {
    "uncalibrated": "prob_uncalibrated",
    "temperature": "prob_temperature",
    "isotonic": "prob_isotonic",
    "pllr": "prob_pllr",
}
y = oof["label"].to_numpy()

rows = []
for name, col in score_cols.items():
    rows.append({"score_variant": name, **full_report(y, oof[col].to_numpy())})
pooled = pd.DataFrame(rows).set_index("score_variant")
pooled.style.format("{:.4f}").background_gradient(cmap="viridis", subset=["roc_auc", "pr_auc", "mcc"])

## Reliability diagram

In [ ]:
fig_path = PROCESSED / f"{GENE}_reliability_notebook.png"
plot_reliability_diagrams(
    series={name.title(): oof[col] for name, col in score_cols.items()},
    y_true=y,
    out_path=str(fig_path),
    n_bins=10,
    title=f"{GENE} — calibrated vs zero-shot PLLR",
)
plt.imshow(plt.imread(fig_path))
plt.axis("off")

## Does the MLP head add signal over the zero-shot PLLR baseline?

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

gain = pd.DataFrame({
    "roc_auc": {name: roc_auc_score(y, oof[col]) for name, col in score_cols.items()},
    "pr_auc": {name: average_precision_score(y, oof[col]) for name, col in score_cols.items()},
})
ax = gain.plot.bar(figsize=(7, 4), rot=0)
ax.set_title(f"{GENE}: learned head vs PLLR zero-shot")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8)

## Held-out VUS prospective predictions

In [ ]:
top = vus.sort_values("confidence_percentile", ascending=False).head(20)
display(top[["hgvs_p", "position", "mut_aa", "mean_calibrated_prob",
             "confidence_percentile", "risk_tier", "fold_std"]])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(vus["mean_calibrated_prob"], bins=25, ax=axes[0])
axes[0].set_title("Calibrated probability distribution (VUS)")
sns.countplot(x=vus["risk_tier"], order=["Low", "Moderate", "High"], ax=axes[1])
axes[1].set_title("Risk tiers")
plt.tight_layout()